# 02 · First Piper fine-tune for Fon (Mac)

Warm-starts a **multi-speaker Piper (VITS medium)** voice from the English `en_US-ljspeech-medium` checkpoint
and fine-tunes it on JML speakers. Text goes in as **characters with tones** (no espeak-ng: it has no Fon).

- `RUN = "tiny"`: a few steps on a small Mac (8 GB), checks that everything runs.
- `RUN = "smoke"`: pipeline check on the M2: data, tone symbols, warm-start, ONNX export, synthesis.
  After a smoke run, expect babble with roughly the right length and rhythm, not Fon.
- `RUN = "full"`: all filtered speakers, on a GPU. The real Kaggle run will get its own notebook.

Why these choices: see the TTS model choice notes (Piper first, warm-start from a public-domain voice, multi-speaker,
only an averaged "population" voice is ever released).

**Licences.** Base voice: LJSpeech data is public domain; the voice was trained from scratch (no lessac).
Training code `piper1-gpl` is GPL-3.0; per its docs/LICENSING.md the GPL covers the engine, not trained voices.

**Setup (once)**
1. `xcode-select --install` and `brew install cmake ninja` (Piper's install compiles espeak-ng, even in text mode).
2. Clone Piper next to this repo and install it into the activated env:
   ```sh
   git clone https://github.com/OHF-Voice/piper1-gpl.git ../piper1-gpl
   uv pip install -e "../piper1-gpl[train]"
   (cd ../piper1-gpl && ./build_monotonic_align.sh)
   ```
   `[train]` pins `librosa<1`, so it may downgrade librosa in this env.
   macOS is not documented by Piper: if the install fails, the tiny run has to move to Kaggle.
3. `python prepare_data.py` (from `kanbior-tts/`) to write `data/train.csv` and `data/val.csv`.

In [ ]:
RUN = "tiny"   # "tiny" (8 GB Mac), "smoke" (M2) or "full" (GPU)

PRESETS = {
    # ~1 epoch of a few steps: checks that everything runs
    "tiny": dict(n_speakers=2, max_train=24, epochs=2, batch_size=4, accelerator="cpu"),
    # pipeline check: 4 speakers, ~50 steps per epoch (time per step on CPU not measured yet)
    "smoke": dict(n_speakers=4, max_train=400, epochs=5, batch_size=8, accelerator="cpu"),
    # all filtered speakers; needs a GPU (Piper has no Apple MPS support)
    "full": dict(n_speakers=None, max_train=None, epochs=1000, batch_size=32, accelerator="auto"),
}
cfg = PRESETS[RUN]

from pathlib import Path

DATA_DIR = Path("../kanbior-asr/sample-data").resolve()   # transcribed audio, shared with the ASR
SPLITS_DIR = Path("data")                                  # written by prepare_data.py
OUTPUT_DIR = Path("runs").resolve() / f"piper-fon-{RUN}"
VOICE_NAME = f"kanbior-fon-{RUN}"
SAMPLE_RATE = 22_050          # Piper medium; JML is 16 kHz, Piper resamples on load (no new high frequencies)
NUM_SYMBOLS = 256             # same as the English checkpoint, so its text embedding loads
SEED = 42

# English base voice: trained from scratch on LJ Speech (public domain)
BASE_REPO = "rhasspy/piper-checkpoints"
BASE_DIR = "en/en_US/ljspeech/medium"
BASE_CKPT = f"{BASE_DIR}/lj-med_1000.ckpt"   # ~846 MB, full training checkpoint
BASE_CONFIG = f"{BASE_DIR}/config.json"

In [ ]:
import csv
import json
import random
import subprocess
import sys
import time
import unicodedata
from collections import Counter

import numpy as np
import torch
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| python", sys.executable)

## 1. English base checkpoint

Download once (cached by huggingface_hub), then check the parts we rely on: 256 text symbols,
a single speaker, 22.05 kHz. The `.ckpt` is a pickle: only load checkpoints from trusted repos.

In [ ]:
base_ckpt = hf_hub_download(BASE_REPO, BASE_CKPT, repo_type="dataset")
base_config = json.loads(Path(hf_hub_download(BASE_REPO, BASE_CONFIG, repo_type="dataset")).read_text())

ckpt = torch.load(base_ckpt, map_location="cpu", weights_only=False)
emb = ckpt["state_dict"]["model_g.enc_p.emb.weight"]
n_gen = sum(v.numel() for k, v in ckpt["state_dict"].items() if k.startswith("model_g."))
print(f"Text embedding: {tuple(emb.shape)}  (symbols x channels)")
print(f"Generator: {n_gen / 1e6:.1f}M parameters | epoch {ckpt.get('epoch')}")
print("Base config:", {k: base_config[k] for k in ("num_symbols", "num_speakers", "phoneme_type")},
      "| sample rate", base_config["audio"]["sample_rate"])
assert emb.shape[0] == NUM_SYMBOLS
del ckpt, emb   # ~850 MB: free it before training

## 2. Data

Multi-speaker: each speaker keeps its own ID during training. Text is cleaned by `prepare_data.py`,
lowercased here (the symbol map has no capitals) and `|` is removed (it is Piper's CSV separator).
Piper splits the text into Unicode code points after NFD, so each tone mark is its own symbol.

Piper makes its own small validation split from `metadata.csv`; `val.csv` stays out of training and is used
for listening in section 5.

In [ ]:
def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def tts_text(text: str) -> str:
    return unicodedata.normalize("NFC", text.lower().replace("|", " ")).strip()

all_train, all_val = read_csv(SPLITS_DIR / "train.csv"), read_csv(SPLITS_DIR / "val.csv")

counts = Counter(r["speaker_id"] for r in all_train)
speakers = sorted(s for s, _ in counts.most_common(cfg["n_speakers"]))   # most_common(None) = all
train_rows = [r for r in all_train if r["speaker_id"] in speakers]
random.Random(SEED).shuffle(train_rows)
train_rows = train_rows[: cfg["max_train"]]
val_rows = [r for r in all_val if r["speaker_id"] in speakers]

metadata_path = OUTPUT_DIR / "metadata.csv"
with open(metadata_path, "w", encoding="utf-8") as f:
    for r in train_rows:   # wav|speaker|text, no header; wav is relative to DATA_DIR
        f.write(f"{r['audio_path']}|{r['speaker_id']}|{tts_text(r['text'])}\n")

missing_audio = [r["audio_path"] for r in train_rows if not (DATA_DIR / r["audio_path"]).is_file()]
assert not missing_audio, f"{len(missing_audio)} audio files missing, e.g. {missing_audio[:3]}"

hours = sum(float(r["duration"]) for r in train_rows) / 3600
print(f"{len(speakers)} speakers, train {len(train_rows)} clips ({hours:.2f} h), "
      f"val for listening {len(val_rows)} clips")
print("Example:", train_rows[0]["speaker_id"], "|", tts_text(train_rows[0]["text"]))

## 3. Fon symbol map

Piper silently **drops** any symbol missing from its map (one log warning each), and its default map has no
tone marks. So we write our own map:
- start from the English voice's map, so shared symbols (a-z, ɛ ɔ ɖ, space, punctuation) keep their ids
  and their trained embeddings;
- `g` reuses the id of IPA `ɡ`: espeak-ng writes /g/ as `ɡ`, so that row is well trained (the English map's plain `g` is rare);
- every other new symbol (mainly the tone marks) gets a free id below 256.

The map is built from **all** of `train.csv` and `val.csv`, not just this run's subset, so ids stay the same
across tiny, smoke and full runs. Then every training sentence is checked against it.

In [ ]:
ALIASES = {"g": "ɡ"}   # Fon letter -> symbol of the English map with the same sound

def symbols(text: str) -> list[str]:
    return list(unicodedata.normalize("NFD", tts_text(text)))   # what Piper does in text mode

fon_map = {k: list(v) for k, v in base_config["phoneme_id_map"].items()}
used = {i for ids in fon_map.values() for i in ids}
free_ids = (i for i in range(NUM_SYMBOLS) if i not in used)

added = []
for ch, target in ALIASES.items():   # overrides the English entry, if any
    fon_map[ch] = fon_map[target]
    added.append((ch, fon_map[ch][0], f"alias of {target}"))
for ch in sorted({ch for r in all_train + all_val for ch in symbols(r["text"])}):
    if ch not in fon_map:
        new_id = next(free_ids)   # StopIteration here means more than 256 symbols
        fon_map[ch] = [new_id]
        added.append((ch, new_id, "new"))

for ch, i, how in added:
    shown = f"◌{ch}" if unicodedata.combining(ch) else ch
    print(f"  {shown:3s} U+{ord(ch):04X} {unicodedata.name(ch, '?'):38s} -> id {i:3d} ({how})")
assert max(i for ids in fon_map.values() for i in ids) < NUM_SYMBOLS

phonemes_path = OUTPUT_DIR / "fon_phonemes.json"
phonemes_path.write_text(json.dumps(fon_map, ensure_ascii=False, indent=1), encoding="utf-8")

# Check: no symbol of this run's text would be dropped
uncovered = Counter(ch for r in train_rows + val_rows for ch in symbols(r["text"]) if ch not in fon_map)
assert not uncovered, f"Symbols missing from the map: {uncovered}"
print(f"\n{len(added)} symbols added or aliased; all {len(train_rows) + len(val_rows)} sentences covered -> {phonemes_path.name}")

## 4. Train

Runs Piper's training in a subprocess through `train_piper.py`: same command line as `python -m piper.train fit`,
but with our checkpoint callbacks (best by `val_mel` + `last.ckpt`). Piper's defaults include one that monitors
`val_mos`, which fails when MOS is off, and they can't be removed from the command line.
Key flags:
- `--model.warmstart_ckpt`: copy every English weight whose shape matches; new parts (speaker embedding)
  start fresh. Not `--ckpt_path`, which resumes the English run with its own settings.
- `--data.phoneme_type text` + `--data.phonemes_path`: characters with tones, our map.
- `--data.espeak_voice en`: required by the CLI, unused in text mode.
- `--model.mos_metric none`: skip the UTMOS download (unreliable on Fon anyway).

The log should show `[warmstart] Copied ... parameters` and no `Missing phoneme from id map`.
Logs, `config.json` and checkpoints go to `runs/piper-fon-<RUN>/`. TensorBoard: `tensorboard --logdir runs`.

In [ ]:
cmd = [
    sys.executable, str(Path("train_piper.py").resolve()), "fit",
    "--data.voice_name", VOICE_NAME,
    "--data.csv_path", str(metadata_path),
    "--data.audio_dir", str(DATA_DIR),
    "--data.cache_dir", str(OUTPUT_DIR / "cache"),
    "--data.config_path", str(OUTPUT_DIR / "config.json"),
    "--data.espeak_voice", "en",
    "--data.phoneme_type", "text",
    "--data.phonemes_path", str(phonemes_path),
    "--data.num_symbols", str(NUM_SYMBOLS),
    "--data.batch_size", str(cfg["batch_size"]),
    "--data.num_workers", "0" if cfg["accelerator"] == "cpu" else "4",
    "--data.num_test_examples", "2",
    "--model.num_speakers", str(len(speakers)),
    "--model.sample_rate", str(SAMPLE_RATE),
    "--model.warmstart_ckpt", base_ckpt,
    "--model.mos_metric", "none",
    "--trainer.accelerator", cfg["accelerator"],
    "--trainer.devices", "1",
    "--trainer.max_epochs", str(cfg["epochs"]),
    "--trainer.log_every_n_steps", "5",
    "--trainer.default_root_dir", str(OUTPUT_DIR),
]
flags = cmd[3:]
print("python train_piper.py fit \\\n    " + " \\\n    ".join(f"{a} {b}" for a, b in zip(flags[::2], flags[1::2])))

In [ ]:
start = time.time()
log_lines = []
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                      env={**__import__("os").environ, "PYTHONUNBUFFERED": "1"}) as proc:
    for line in proc.stdout:
        log_lines.append(line)
        if "it/s]" not in line and "s/it]" not in line:   # skip progress-bar redraws
            print(line, end="")
print(f"\nExit code {proc.returncode}, {(time.time() - start) / 60:.1f} min")

log = "".join(log_lines)
(OUTPUT_DIR / "train.log").write_text(log, encoding="utf-8")
assert proc.returncode == 0, "training failed: see the log above"
assert "[warmstart] Copied" in log, "warm-start did not run"
assert "Missing phoneme from id map" not in log, "some symbols were dropped: check the map"

## 5. Export and listen

Exports the last checkpoint to ONNX (what production would run, with onnxruntime) and synthesizes
`val.csv` sentences for each speaker, next to the real recording.

These per-speaker voices are for checking only; they reproduce volunteers' voices and are never released.
The released voice will be an average of the speaker embeddings (a later step).

In [ ]:
versions = sorted((OUTPUT_DIR / "lightning_logs").glob("version_*"), key=lambda p: int(p.name.split("_")[1]))
last_ckpt = versions[-1] / "checkpoints" / "last.ckpt"
onnx_path = OUTPUT_DIR / f"{VOICE_NAME}-medium.onnx"
subprocess.run([sys.executable, "-m", "piper.train.export_onnx",
                "--checkpoint", str(last_ckpt), "--output-file", str(onnx_path)], check=True)
onnx_config = Path(f"{onnx_path}.json")
onnx_config.write_text((OUTPUT_DIR / "config.json").read_text(encoding="utf-8"), encoding="utf-8")
print(f"{onnx_path.name}: {onnx_path.stat().st_size / 1e6:.1f} MB (from {last_ckpt.parent.parent.name})")

In [ ]:
from piper import PiperVoice, SynthesisConfig

voice = PiperVoice.load(onnx_path, config_path=onnx_config)
speaker_ids = voice.config.speaker_id_map

# One sentence per speaker; tiny speakers may have no val clip, then use a training sentence
listen = [next((r for r in val_rows if r["speaker_id"] == s), None) or
          next(r for r in train_rows if r["speaker_id"] == s) for s in speakers]

gen_time = gen_audio = 0.0
for r in listen:
    print(f"\n{r['id']}  {r['speaker_id']}{'' if r in val_rows else ' (train sentence)'}:  {r['text']}")
    print("  reference:")
    display(Audio(str(DATA_DIR / r["audio_path"])))
    t = time.time()
    chunks = list(voice.synthesize(tts_text(r["text"]), SynthesisConfig(speaker_id=speaker_ids[r["speaker_id"]])))
    gen_time += time.time() - t
    wav = np.concatenate([c.audio_float_array for c in chunks])
    gen_audio += len(wav) / chunks[0].sample_rate
    print("  synthesized:")
    display(Audio(wav, rate=chunks[0].sample_rate))

print(f"\nReal-time factor on this CPU: {gen_time / gen_audio:.3f} (below 1 = faster than real time)")